Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

Load the Breast Cancer Dataset

In [ ]:
# load the dataset

data = load_breast_cancer()

X = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

y = pd.Series(
    (data.target == 0).astype(int),
    name="malignant"
)

print(y.value_counts())

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Class names:", data.target_names)

Examine the Class Distribution

In [ ]:
# examine the class distribution

class_counts = y.value_counts().sort_index()

class_distribution = pd.DataFrame({
    "Class": data.target_names,
    "count": class_counts.values,
    "Probability": class_counts.values / len(y)
})

print(class_distribution)

Prepare Class Distribution for Plotting

In [ ]:
class_distribution = y.value_counts().reset_index()

class_distribution.columns = ["Class", "Count"]

Visualize Class Distribution

In [ ]:
class_distribution.plot(
    x="Class",
    y="Count",
    kind="bar",
    legend=False,
    color=["tomato", "steelblue"]
)

plt.ylabel("Number of observations")

plt.title("Class Distribution")

plt.xticks(rotation=0)

plt.show()

Visualize Class Distribution as a Pie Chart

In [ ]:
class_distribution.plot(
    x="Class",
    y="Count",
    kind="pie",
    legend=False,
    color=["tomato", "steelblue"]
)

plt.ylabel("Number of observations")

plt.title("Class Distribution")

plt.xticks(rotation=0)

plt.show()

Split the Dataset into Training and Testing Sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training size:", len(y_train))

print("Testing size:", len(y_test))

print("\n Training proportions:")

print(y_train.value_counts(normalize=True).sort_index())

print("\n Testing proportions:")

print(y_test.value_counts(normalize=True).sort_index())

Create and Train the Logistic Regression Model

In [ ]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

model.fit(X_train, y_train)

Obtain Predicted Probabilities

In [ ]:
#obtain predicted probabilities

probabilities = model.predict_proba(X_test)

print(probabilities[:5])

Create Prediction Results DataFrame

In [ ]:
results = pd.DataFrame({
    "Actual_class": y_test.values,
    "P_malignant": probabilities[:, 0],
    "P_benign": probabilities[:, 1]
})

print(results.head(10))

Map Actual Classes to Labels

In [ ]:
results["Actual_label"] = results["Actual_class"].map({
    0: "Malignant",
    1: "Benign"
})



In [ ]:
results = pd.DataFrame({
    "Actual_class": y_test.values,
    "P_malignant": probabilities[:, 0],
    "P_benign": probabilities[:, 1]
})

Display Actual Labels and Probabilities

In [ ]:
print(
    results[
        ["Actual_label", "P_malignant", "P_benign"]
    ].head(10)
)

Apply Classification Threshold

In [ ]:
threshold = 0.50

results["Predicted_malignant"] = (
    results["P_malignant"] >= threshold
).astype(int)

results["Predicted_label"] = results[
    "Predicted_malignant"
].map({
    1: "Malignant",
    0: "Benign"
})

print(
    results[
        ["Actual_class", "P_malignant", "Predicted_label"]
    ].head(10)
)

Compare Different Thresholds

In [ ]:
# Compare different thresholds

for threshold in [0.30, 0.50, 0.70]:
    predictions = (
        results["P_malignant"] >= threshold
    ).astype(int)

    print(
        f"Threshold = {threshold}: "
        f"Predicted malignant cases = {predictions.sum()}"
    )

Construct Confusion Matrix

In [ ]:
# Construct a confusion matrix

actual_malignant = (y_test.values == 0).astype(int)

for threshold in [0.30, 0.50, 0.70]:
    predicted_malignant = (
        probabilities[:, 0] >= threshold
    ).astype(int)

    cm = confusion_matrix(
        actual_malignant,
        predicted_malignant
    )

    print(f"\nThreshold = {threshold}")
    print(cm)

Verify Accuracy Using Scikit-learn

In [ ]:
# Predict malignant if probability >= threshold

y_pred = (
    results["P_malignant"] >= threshold
).astype(int)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print(
    "Sklearn accuracy:",
    accuracy_score(y_test, y_pred)
)

Calculate Metrics Across Thresholds

In [ ]:
thresholds = [0.1, 0.3, 0.5, 0.7, 0.9]

threshold_metrics = []

for threshold in thresholds:

    y_pred = (
        results["P_malignant"] >= threshold
    ).astype(int)

    # Confusion Matrix
    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_test, y_pred)

    threshold_metrics.append({
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Accuracy": accuracy,
        "Precision": precision,
        "Sensitivity_Recall": recall,
        "Specificity": specificity,
        "F1_Score": f1
    })

threshold_metrics = pd.DataFrame(threshold_metrics)

print(threshold_metrics.round(3))